# 随机数与可复现采样

学习目标：生成指定形状的随机样本，完成有无放回抽样与顺序打乱，说明复现条件并检查样本约束。

前置知识：数组形状、索引、频数、均值、概率分布的基本含义。

运行环境：Python 3.12、NumPy 2.5.3；本章使用的默认位生成器为 PCG64。随机输出对应当前环境，不承诺跨版本逐位一致。

环境准备：见 [环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

首个代码单元导入 NumPy，后续单元沿用 np。

标明“预期异常”的单元会直接显示原始报错；阅读异常类型与原因后，继续运行下一单元。

## 1 随机抽取设备

从五台设备中抽取三台检查，用 choice 从编号数组中选择元素。replace=False 表示无放回，同一个位置不会再次入选；这里的编号本身也互不重复。

default_rng 创建局部随机数生成器，2026 是本例的种子。后面的随机操作通过 rng 调用，便于明确哪一次实验在使用这段随机状态。

In [1]:
import numpy as np

device_ids = np.array([101, 102, 103, 104, 105])
rng = np.random.default_rng(2026)
selected = rng.choice(device_ids, size=3, replace=False)

print(selected)  # 本次抽取三个合法编号，次序为实际采样结果。
print(selected.shape, selected.dtype)  # (3,)，本机为 int64。
print(np.unique(selected).size == 3)  # True：三个编号互不重复。

[105 103 101]
(3,) int64
True


## 2 生成器与样本形状

随机操作由两层配合完成。一般采样任务直接使用 Generator；需要明确底层算法时，才指定 BitGenerator。

| 名称 | 中文名称／含义 | 作用 |
| --- | --- | --- |
| default_rng | 默认生成器构造函数 | 用种子创建 Generator；不管理全局随机实例 |
| Generator | 随机数生成器 | 提供整数、正态分布、抽样等接口 |
| BitGenerator | 位生成器 | 管理状态并产生随机比特，供 Generator 使用 |
| PCG64 | 一种位生成器 | NumPy 2.5 中 default_rng 默认使用的算法 |

未提供种子时，default_rng 使用操作系统提供的不可预测数据。需要重复实验时，显式给出种子。多数采样接口的 size 省略时返回一个样本，整数 size 表示一维样本数，元组 size 表示输出形状。

In [2]:
rng = np.random.default_rng(2026)
print(type(rng).__name__, type(rng.bit_generator).__name__)  # Generator PCG64

# 明确指定算法；同一 PCG64 种子可交给 Generator 使用。
explicit_rng = np.random.Generator(np.random.PCG64(2026))
print(explicit_rng.integers(0, 10))  # 本次抽到的单个整数，范围为 0 到 9。
print(explicit_rng.integers(0, 10, size=3).shape)  # (3,)
print(explicit_rng.integers(0, 10, size=(2, 3)).shape)  # (2, 3)

Generator PCG64
8
(3,)
(2, 3)


## 3 整数、均匀与正态采样

### 3.1 整数端点

为两组设备各生成三次整数档位，输出形状为 (2, 3)：第 0 轴是组，第 1 轴是采样次数。

integers 的 low 包含在范围内，high 默认不包含。endpoint=True 改为包含 high。只传一个边界时，它作为上界，下界为 0。默认输出类型为 int64。

In [3]:
rng = np.random.default_rng(2026)
levels = rng.integers(1, 4, size=(2, 3))
inclusive = rng.integers(1, 4, size=(2, 3), endpoint=True)

print(levels)  # 本次整数样本，形状为 (2, 3)，值为 1 到 3。
print(levels.shape, levels.dtype)  # (2, 3)，int64。
print(np.all((levels >= 1) & (levels < 4)))  # True：只能取 1、2、3。
print(inclusive)  # 本次整数样本，允许取 1 到 4。
print(np.all((inclusive >= 1) & (inclusive <= 4)))  # True：允许取到 4。
# 允许出现端点，不表示这个小样本一定包含端点。

[[3 1 1]
 [2 2 2]]
(2, 3) int64
True
[[1 2 3]
 [2 4 4]]
True


### 3.2 均匀采样

uniform 用 low 和 high 描述均匀采样区间，要求 high 不小于 low。下面模拟两组设备各三次在 18 到 22 之间的温度读数。

名义区间为左闭右开，但浮点运算的舍入可能使结果取到 high。边界检查要考虑这一实现条件；low 等于 high 时返回该边界值。

In [4]:
rng = np.random.default_rng(2026)
temperatures = rng.uniform(18.0, 22.0, size=(2, 3))

print(temperatures)  # 本次均匀样本，两组各三次读数。
print(temperatures.shape)  # (2, 3)
print(np.all((temperatures >= 18.0) & (temperatures <= 22.0)))
print(rng.uniform(20.0, 20.0, size=3))  # 三个值均为 20。

[[18.71573925 20.55965266 19.8690736 ]
 [19.48200211 19.41966934 21.16207298]]
(2, 3)
True
[20. 20. 20.]


### 3.3 正态采样

normal 用 loc 指定分布均值，用 scale 指定标准差。scale 必须非负，它不是方差，也不是样本必须落入的区间半宽。下面用均值 20、标准差 0.5 的正态分布模拟测量读数。

In [5]:
rng = np.random.default_rng(2026)
readings = rng.normal(loc=20.0, scale=0.5, size=(2, 3))
print(readings)  # 本次读数或排列结果，形状见下方检查。
print(readings.shape)  # (2, 3)：两组，每组三次。

# 预期 ValueError：负标准差不合法。
rng.normal(loc=20.0, scale=-0.5, size=3)

[[19.60343876 20.12028564 19.05183683]
 [20.69788586 20.31914737 19.85397626]]
(2, 3)


ValueError: scale < 0

## 4 抽样与概率权重

### 4.1 有放回与无放回

choice 默认 replace=True，选过的位置可以再次入选。replace=False 则不重复选择位置，样本数不能超过可选位置数。如果原数组本身有重复值，无放回也不保证结果值互不重复。

In [6]:
device_ids = np.array([101, 102, 103])
rng = np.random.default_rng(2026)
repeated = rng.choice(device_ids, size=8, replace=True)
distinct = rng.choice(device_ids, size=3, replace=False)

print(repeated)  # 本次八个样本，只从给定的三个编号中选择。
print(np.unique(repeated).size < repeated.size)  # True：8 次选择只有 3 种编号。
print(distinct)  # 本次无放回抽样，形状为 (3,)。
print(np.unique(distinct).size == 3)  # True：三个位置各选一次。

# 预期 ValueError：无放回时不能从 3 个位置选 4 个。
rng.choice(device_ids, size=4, replace=False)

[103 101 101 102 102 102 101 102]
True
[101 102 103]
True


ValueError: Cannot take a larger sample than population when replace is False

### 4.2 按权重选择

设备被抽检的机会不相同时，先把非负权重除以权重总和，得到概率向量 p。p 必须是一维，与可选元素一一对应，所有概率非负且总和为 1；权重总和必须大于 0。

下面给三台设备设置 1∶2∶7 的权重。有放回采样允许反复抽到同一设备。概率决定抽样机制，不保证有限样本的频率恰好等于概率。

In [7]:
device_ids = np.array([101, 102, 103])
weights = np.array([1.0, 2.0, 7.0])
probabilities = weights / weights.sum(dtype=np.float64)
rng = np.random.default_rng(2026)
selected = rng.choice(device_ids, size=20, replace=True, p=probabilities)
ids, counts = np.unique(selected, return_counts=True)

print(probabilities)  # [0.1 0.2 0.7]
print(selected)  # 本次二十个有放回样本。
print(ids, counts)  # unique 仅列出本次出现过的编号，未出现者频数为 0。
print(counts / selected.size)  # 本次观察频率，不要求与概率精确相等。

[0.1 0.2 0.7]
[102 103 103 103 103 103 103 102 103 102 103 103 103 103 103 103 103 103
 102 102]
[102 103] [ 5 15]
[0.25 0.75]


原始权重不能直接当作概率传给 p。带概率的无放回抽样还要求正概率位置足够多，概率为 0 的位置不能用于补足样本数。

In [8]:
device_ids = np.array([101, 102, 103])
rng = np.random.default_rng(2026)

# 预期 ValueError：概率总和不是 1。
rng.choice(device_ids, size=2, p=[1.0, 2.0, 7.0])

ValueError: Probabilities do not sum to 1. See Notes section of docstring for more information.

In [9]:
# 预期 ValueError：只有一个正概率位置。
rng.choice(device_ids, size=2, replace=False, p=[1.0, 0.0, 0.0])

ValueError: Fewer non-zero entries in p than size

## 5 打乱顺序

shuffle 直接修改传入数组，返回 None。permutation 返回打乱后的副本，保留输入数组。两者对多维数组默认沿 axis=0 操作；下面每行是一台设备的两次读数，因此整行一起移动，行内顺序保持。

In [10]:
# 1. shuffle：直接修改输入数组。
readings = np.array([[10, 11], [20, 21], [30, 31]])
rng = np.random.default_rng(2026)
result = rng.shuffle(readings, axis=0)

print(result)  # None：结果已经写回 readings。
print(readings)  # 行整体重排，行内两个读数的顺序保持。
print(readings.shape)  # (3, 2)
print(np.all(readings[:, 1] - readings[:, 0] == 1))  # True：每行仍成对。

# 2. permutation：重新准备相同输入与种子，观察返回副本的方式。
readings = np.array([[10, 11], [20, 21], [30, 31]])
rng = np.random.default_rng(2026)
shuffled = rng.permutation(readings, axis=0)

print(readings)  # 原顺序保留。
print(shuffled)  # 本次随机行顺序，形状为 (3, 2)。
print(np.shares_memory(readings, shuffled))  # False：副本不共享数据。
print(rng.permutation(4))  # 将 0、1、2、3 排成一个随机顺序。
# 随机排列允许恰好保持原顺序，不能以“顺序必须变化”作为检查条件。

None
[[20 21]
 [10 11]
 [30 31]]
(3, 2)
True
[[10 11]
 [20 21]
 [30 31]]
[[20 21]
 [10 11]
 [30 31]]
False
[0 1 2 3]


## 6 种子与复现条件

种子用于初始化状态，每次采样都会推进状态。复现一次实验，需要记录 NumPy 版本、位生成器、种子、输入数据，以及完整的调用顺序和参数；size 也属于参数。本章保存输出使用 NumPy 2.5.3 和 PCG64。

下面分别从同一种子开始，按“先抽设备，再生成噪声”的顺序运行两次。相同环境下，对应结果一致。若只从中间单元重新运行，已有 rng 的状态可能已经改变。

In [11]:
device_ids = np.array([101, 102, 103, 104, 105])
first_rng = np.random.Generator(np.random.PCG64(2026))
first_ids = first_rng.choice(device_ids, size=3, replace=False)
first_noise = first_rng.normal(0.0, 0.1, size=3)

second_rng = np.random.Generator(np.random.PCG64(2026))
second_ids = second_rng.choice(device_ids, size=3, replace=False)
second_noise = second_rng.normal(0.0, 0.1, size=3)

print(first_ids, first_noise)  # 本次三个编号及对应的三个噪声样本。
print(np.array_equal(first_ids, second_ids))  # True
print(np.array_equal(first_noise, second_noise))  # True

[105 103 101] [ 0.13957717  0.06382947 -0.02920475]
True
True


连续使用同一个生成器是在继续抽样；重新创建同种子生成器是在重新开始。不能在每次需要新样本时都重新设置同一个种子。

Generator 不保证跨版本逐位一致。严格的随机流一致性还依赖相同构建、环境和机器等条件；某些分布会受底层数值库影响。即使种子相同，改变调用顺序、采样方法或批量大小，也不应假设仍得到原序列。

In [12]:
rng = np.random.default_rng(2026)
first = rng.integers(0, 100, size=4)
following = rng.integers(0, 100, size=4)
restarted = np.random.default_rng(2026).integers(0, 100, size=4)

print(first)  # 本次四个整数样本，范围为 0 到 99。
print(following)  # 从后续状态继续抽样，不要求每个值都不同。
print(np.array_equal(first, restarted))  # True：相同条件下重现开头一段。

[85 17  2 63]
[36 46  7 37]
True


## 7 样本统计与理论参数

分布参数描述生成样本的模型，样本均值和样本标准差来自这一次有限观测。即使固定种子，样本均值也不必等于 loc，样本标准差也不必等于 scale。

下面生成 1000 个读数，只打印前六个和统计摘要。std 使用 ddof=0，描述这批数据本身的离散程度；不把与理论值精确相等作为采样正确的条件。

In [13]:
rng = np.random.default_rng(2026)
readings = rng.normal(loc=20.0, scale=0.5, size=1000)

print(readings[:6])  # 本次一千个正态读数中的前六个。
print(readings.shape, readings.dtype)  # (1000,)，浮点样本。
print(readings.mean())  # 本次样本均值。
print(readings.std(ddof=0))  # 本次样本标准差，ddof 为 0。
# 两个统计量仅是本次实测结果，不预设固定误差或要求等于 20、0.5。

[19.60343876 20.12028564 19.05183683 20.69788586 20.31914737 19.85397626]
(1000,) float64
20.010533609944854
0.5152278032825302


## 8 选学：其他分布
按任务含义选择分布，并先核对参数条件。

| 方法 | 中文名称／含义 | 本例参数条件 |
| --- | --- | --- |
| binomial | 二项分布 | n 为非负整数试验次数，成功概率 p 在 0 到 1 之间 |
| poisson | 泊松分布 | lam 为固定区间内的期望事件数，必须非负 |
| multivariate_normal | 多元正态分布 | mean 是均值向量，cov 是对应的对称半正定协方差矩阵 |

二项样本可以表示每批检查十件产品的合格数；泊松样本可以表示每小时出现的事件数。这里仅模拟数据，不据此认定现实任务一定服从这些分布。

In [14]:
rng = np.random.default_rng(2026)
success_counts = rng.binomial(n=10, p=0.8, size=5)
event_counts = rng.poisson(lam=2.0, size=5)

print(success_counts)  # 每个值都是 0 到 10 之间的整数。
print(event_counts)  # 每个值都是非负整数，不限定为 2。
print(success_counts.shape, event_counts.shape)  # 均为 (5,)

[9 8 8 9 9]
[2 4 4 1 3]
(5,) (5,)


两个通道一起模拟时，mean 的长度决定通道数，cov 的对角线是各通道方差。下面用对角协方差矩阵，三个样本形成 (3, 2) 数组。check_valid="raise" 要求发现无效协方差时抛出异常。

In [15]:
rng = np.random.default_rng(2026)
samples = rng.multivariate_normal(
    mean=[0.0, 0.0], cov=[[1.0, 0.0], [0.0, 4.0]],
    size=3, check_valid="raise",
)
print(samples)  # 本次多元正态样本，三个样本、两个通道。
print(samples.shape)  # (3, 2)：三个样本，每个样本两个通道。

# 预期 ValueError：负方差使这个矩阵无效。
rng.multivariate_normal(
    mean=[0.0, 0.0], cov=[[1.0, 0.0], [0.0, -1.0]],
    size=3, check_valid="raise",
)

[[ 0.24057128 -1.58624495]
 [ 1.39577171 -3.7926527 ]
 [-0.29204749  1.27658948]]


(3, 2)


ValueError: covariance is not symmetric positive-semidefinite.

## 9 选学：状态与派生流
### 9.1 保存中途状态

种子重建起点，状态快照用于从中途继续。BitGenerator 的 state 属性可读取或设置状态字典。下面使用 deepcopy 保存独立快照，再恢复到相同位生成器的另一个实例；示例只在当前环境内恢复，不作为跨版本存档方案。

In [16]:
from copy import deepcopy

rng = np.random.Generator(np.random.PCG64(2026))
print(rng.integers(0, 10, size=3))  # 先消费一段序列。
saved_state = deepcopy(rng.bit_generator.state)
expected = rng.integers(0, 10, size=4)

restored_rng = np.random.Generator(np.random.PCG64(0))
restored_rng.bit_generator.state = saved_state
restored = restored_rng.integers(0, 10, size=4)
print(expected, restored)  # 两组都是形状为 (4,) 的整数样本。
print(np.array_equal(expected, restored))  # True：接着快照位置继续。

[8 1 0]
[6 3 4 0] [6 3 4 0]
True


### 9.2 为不同任务派生生成器

需要给多个任务各自的生成器时，用 SeedSequence.spawn 派生子种子，再分别创建 Generator。官方将这些子流描述为以很高概率彼此独立；几次输出不同不能证明独立性。

复现时，保留根种子、派生树结构、子流与任务的对应关系以及各自的调用顺序，不把同一个种子反复交给不同任务作为独立流。

In [17]:
seed_sequence = np.random.SeedSequence(2026)
child_seeds = seed_sequence.spawn(2)
inspection_rng = np.random.default_rng(child_seeds[0])
measurement_rng = np.random.default_rng(child_seeds[1])

print(inspection_rng.integers(0, 10, size=4))  # 抽检子流的本次样本，值为 0 到 9。
print(measurement_rng.integers(0, 10, size=4))  # 测量子流的本次样本，值为 0 到 9。
# 两个生成器分别推进自己的状态，输出差异不是独立性的检验。

[6 3 1 0]
[7 6 6 1]


## 10 选学：旧接口的兼容用途
RandomState 是旧随机接口，主要用于需要延续旧代码随机序列的场景；其默认位生成器为 MT19937。新代码通常使用 default_rng。两个接口即使种子相同，也不应假设分布采样序列相同。

下面仅识别旧写法，使用局部实例，不修改全局随机状态。RandomState 的序列兼容承诺也有构建与平台条件，不能代替环境记录。

In [18]:
legacy_rng = np.random.RandomState(2026)
print(legacy_rng.normal(loc=20.0, scale=0.5, size=3))  # 旧接口的本次三个正态样本。
# 兼容旧程序时保留旧接口和调用过程，不直接换 API 后期待原序列。

[19.78414074 19.30356302 20.15578533]


## 本章小结

（1）default_rng 创建局部 Generator；Generator 提供采样接口，BitGenerator 管理随机状态与比特。

（2）整数端点、样本形状、概率归一化和放回条件先决定合法输入，再决定怎样检查结果。

（3）shuffle 原地打乱，permutation 返回副本；多维输入要明确打乱的轴。

（4）复现需要记录环境、算法、种子和完整调用过程；样本统计是观测值，不是理论参数的精确副本。

## 练习

（1）改变抽检规则

先从五台设备中抽检三台，要求设备不重复。随后任务改为连续安排八次检查，允许重复，权重为 1∶1∶2∶2∶4。分别选择参数并说明理由，解释第二个任务为什么不能继续使用原来的放回设置。

In [19]:
import numpy as np

device_ids = np.array([101, 102, 103, 104, 105])
weights = np.array([1.0, 1.0, 2.0, 2.0, 4.0])
rng = np.random.default_rng(42)

# 在此完成两次抽样，打印形状并核对第一次的唯一编号数。
# 在此打印概率向量的总和，并用注释说明两次 replace 设置的理由。

（2）预测端点与形状

运行前写下两个数组的形状、可能的最小值和最大值。再运行并解释：如果本次没有抽到允许的最大值，是否说明端点设置失效？

In [20]:
rng = np.random.default_rng(42)
first = rng.integers(2, 5, size=(2, 3))
second = rng.integers(2, 5, size=4, endpoint=True)

# 在此先写预测，再打印数组与形状；用范围条件检查，不要求端点必然出现。

（3）保留记录顺序

每行是一台设备的两次读数。请随机调整设备的检查顺序，同时保留原数组和每行内部顺序。选择 shuffle 或 permutation 并说明理由；如果允许覆盖原数组，你会怎样修改？

In [21]:
readings = np.array([[10, 11], [20, 21], [30, 31], [40, 41]])
rng = np.random.default_rng(42)

# 在此生成随机行顺序，打印原数组与结果，核对形状和每行的差值。
# 在此用注释说明选择的方法以及覆盖原数组时的替代写法。

（4）重复完整采样过程

先无放回选择两个设备编号，再为它们生成均值为 0、标准差为 0.2 的两个噪声值。从同一种子重建生成器后，完整重做这两个操作并比较结果。再交换调用顺序，说明为什么不能只根据种子相同就承诺原结果。

In [22]:
device_ids = np.array([101, 102, 103, 104])
seed = 42

# 在此显式使用 PCG64，保存两轮对应结果并用 array_equal 比较。
# 在此另外创建生成器交换调用顺序，观察结果并解释复现所需条件。
# 在此用注释记录版本条件；不把“本次不同”写成所有样本都必然不同。

### 重点练习提示

对应第（1）题。先独立完成，再按需要查看提示。

（1）先比较要抽取的数量与设备总数，再决定是否允许同一设备再次被抽中。

（2）权重需要除以总和变成概率；抽样验收检查范围、形状和去重约束，不要求每次频数等于概率比例。

### 重点练习参考解析

对应第（1）题。

第一次 choice 设置 size=3、replace=False，不指定 p 即按等概率抽取，结果形状为 (3,)，三个编号必须不同。第二次设置 size=8、replace=True，概率由 weights/weights.sum() 得到 [0.1, 0.1, 0.2, 0.2, 0.4]，总和在浮点容差内为 1。

只有 5 台设备，无法无放回抽取 8 次，因此第二次必须允许重复。其结果形状为 (8,)，所有编号都来自输入；一次小样本的频数不必符合 1∶1∶2∶2∶4。检查这些条件即可，不把具体抽样序列作为跨环境固定答案。

## 参考与引用来源

| 网站 | 本章参考内容与定位 |
| --- | --- |
| NumPy 官方文档（2.5） | [Random Generator](https://numpy.org/doc/2.5/reference/random/generator.html)：Generator、BitGenerator、default_rng 与无版本兼容保证；[integers](https://numpy.org/doc/2.5/reference/random/generated/numpy.random.Generator.integers.html)、[uniform](https://numpy.org/doc/2.5/reference/random/generated/numpy.random.Generator.uniform.html)、[normal](https://numpy.org/doc/2.5/reference/random/generated/numpy.random.Generator.normal.html)：Parameters、Returns 与 uniform 的浮点端点说明；[choice](https://numpy.org/doc/2.5/reference/random/generated/numpy.random.Generator.choice.html)：replace、p、Raises 与 Notes；[shuffle](https://numpy.org/doc/2.5/reference/random/generated/numpy.random.Generator.shuffle.html)、[permutation](https://numpy.org/doc/2.5/reference/random/generated/numpy.random.Generator.permutation.html)：原地修改、副本与 axis；[Compatibility policy](https://numpy.org/doc/2.5/reference/random/compatibility.html)：相同随机流的严格条件、调用大小和跨版本限制；[binomial](https://numpy.org/doc/2.5/reference/random/generated/numpy.random.Generator.binomial.html)、[poisson](https://numpy.org/doc/2.5/reference/random/generated/numpy.random.Generator.poisson.html)、[multivariate_normal](https://numpy.org/doc/2.5/reference/random/generated/numpy.random.Generator.multivariate_normal.html)：参数条件与返回形状；[BitGenerator.state](https://numpy.org/doc/2.5/reference/random/bit_generators/generated/numpy.random.BitGenerator.state.html)：状态字典；[Parallel random number generation](https://numpy.org/doc/2.5/reference/random/parallel.html)：SeedSequence spawning；[Legacy random generation](https://numpy.org/doc/2.5/reference/random/legacy.html)：RandomState 兼容用途；[unique](https://numpy.org/doc/2.5/reference/generated/numpy.unique.html)、[mean](https://numpy.org/doc/2.5/reference/generated/numpy.mean.html)、[std](https://numpy.org/doc/2.5/reference/generated/numpy.std.html)：计数与样本统计口径；[shares_memory](https://numpy.org/doc/2.5/reference/generated/numpy.shares_memory.html)：检查数组数据是否共享。 |
| NumPy 官方源码（GitHub，v2.5.0） | [Generator.choice 实现](https://github.com/numpy/numpy/blob/v2.5.0/numpy/random/_generator.pyx)：choice 的 p 参数检查与 replace=False 分支，正概率位置数不足时抛出 ValueError。 |
| Python 官方文档（3.12） | [copy](https://docs.python.org/3.12/library/copy.html)：deepcopy 对复合对象递归复制，用于保存独立状态快照。 |